# 00 — Derive JUMP-Target2 manifest

Read pipeline metadata, intersect compound sets across sources for the
`TARGET2` plate type, cache the resulting compound IDs to JSON.

**First run**: column-name guesses may be wrong. The helper raises with
the available `Metadata_*` columns listed — paste the right ones below.

In [1]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))
from src.target2 import derive_target2_manifest, cache_manifest
from src.paths import PIPELINE_OUT, DATA_OUT, scenario_method_parquet

## Pick a metadata source

Any pipeline parquet works — they all carry the same `Metadata_*` columns.
We use the largest available scenario for maximum source coverage.

In [2]:
# Edit if the chosen scenario doesn't have all sources.
META_SOURCE = scenario_method_parquet("scenario_5", "scvi_single")
assert META_SOURCE.exists(), f"missing {META_SOURCE} — pick another scenario"
META_SOURCE

PosixPath('/ictstr01/groups/ml01/workspace/ttreis/projects/broad_integrate/2023_Arevalo_BatchCorrection/outputs/scenario_5/mad_int_featselect_scvi_single.parquet')

## Inspect available `Metadata_*` columns

Run this if column-name guesses fail.

In [3]:
import pandas as pd
head = pd.read_parquet(META_SOURCE).head(0)
[c for c in head.columns if c.startswith("Metadata_")]

['Metadata_Source',
 'Metadata_Plate',
 'Metadata_Well',
 'Metadata_JCP2022',
 'Metadata_InChIKey',
 'Metadata_InChI',
 'Metadata_Batch',
 'Metadata_PlateType',
 'Metadata_PertType',
 'Metadata_Row',
 'Metadata_Column',
 'Metadata_Microscope']

## Derive the manifest

Override `plate_col` / `source_col` / `compound_col` if defaults don't
match what you found above.

In [4]:
target2 = derive_target2_manifest(META_SOURCE)
print(f"Found {len(target2)} Target2 compounds")
sorted(target2)[:5]

Found 301 Target2 compounds


/ictstr01/groups/ml01/workspace/ttreis/projects/broad_integrate/2023_Arevalo_BatchCorrection/reference_mapping/src/target2.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  by_source = target2_plates.groupby(source_col)[compound_col].apply(set)


['AMG900', 'Aloxistatin', 'DMSO', 'Dexamethasone', 'FK-866']

## Cache to JSON

In [5]:
cache_path = DATA_OUT / "target2_compounds.json"
cache_manifest(target2, cache_path)
print(f"Wrote {cache_path}")

Wrote /ictstr01/groups/ml01/workspace/ttreis/projects/broad_integrate/2023_Arevalo_BatchCorrection/reference_mapping/data/target2_compounds.json
